In [ ]:
import glob
import os

import pandas as pd
import numpy as np

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

import napari

import matplotlib.pyplot as plt
import matplotlib

from tqdm import tqdm

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [21]:
# Import images

img_raw_dir = '/Volumes/bamfaile-1/Franklin/20241010/ACTB-KI_NGN2_Batch20241005_D5/50ms/Selected_for_analysis'
img_raw_path = os.path.join(img_raw_dir,'*.stk')
img_raw_files = np.sort(glob.glob(img_raw_path))

# Extract the integers before the first underscore in the filenames
loaded_image_ids = {
    os.path.basename(f).split('_')[0]  # Extract the part before the first underscore
    for f in img_raw_files
}

img_denoised_dir = '/Volumes/bamfaile/Gitrepos_temp2/gchao_singlemolecule_tracking/runs/iNeurons/iNeurons_unperturbed/03_Predict/20241010'
img_denoised_path = os.path.join(img_denoised_dir,'*.stk') 
img_denoised_files = np.sort(glob.glob(img_denoised_path))

# Filter the new images based on matching IDs
img_denoised_files = [
    f for f in img_denoised_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
]


In [22]:
# Output directory

img_composite_dir = r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D5/Unperturbed/50ms/Denoised_merged'

In [ ]:
img_raw_files

In [ ]:
print(len(img_denoised_files))
print(len(img_raw_files))

In [ ]:
# Read images into list

img_denoised = []
img_raw = []

for file in tqdm(img_denoised_files):
    image = imread(file)
    img_denoised.append(image)

for file in tqdm(img_raw_files):
    image = imread(file)
    img_raw.append(image)

In [ ]:
print(img_denoised[0].dtype, img_denoised[0].shape, np.min(img_denoised[0]), np.max(img_denoised[0]))

In [ ]:
# Plot raw and denoised image next to each other

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(img_denoised[0][0])
ax[1].imshow(img_raw[0][0])

In [ ]:
# Load actin images
images_actin_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D5/Unperturbed/Center_slice/Actin'
images_actin_path = os.path.join(images_actin_dir, '*.tif')
image_actin_files = np.sort(glob.glob(images_actin_path))

# Filter the new images based on matching IDs
image_actin_files = [
    f for f in image_actin_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
]

# Load tubulin images
images_tubulin_dir = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241005_iNs/D5/Unperturbed/Center_slice/Tubulin'
images_tubulin_path = os.path.join(images_tubulin_dir, '*.tif')
image_tubulin_files = np.sort(glob.glob(images_tubulin_path))

# Filter the new images based on matching IDs
image_tubulin_files = [
    f for f in image_tubulin_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
]

# Print or process the filtered new images
print(len(image_actin_files))
print(len(image_tubulin_files))

In [ ]:
# Read images into list

img_actin = []
img_tubulin = []

for file in tqdm(image_actin_files):
    image = imread(file)
    img_actin.append(image)

for file in tqdm(image_tubulin_files):
    image = imread(file)
    img_tubulin.append(image)

In [ ]:
print(len(img_denoised))
print(len(img_raw))
print(len(img_actin))
print(len(img_tubulin))

## Multiplying actin and tubulin images to create artificial series

In [ ]:
# Example: Assuming img_actin is a list of 2D numpy arrays (single slices)
img_actin_series = [np.repeat(img[np.newaxis, ...], 100, axis=0) for img in img_actin]
img_tubulin_series = [np.repeat(img[np.newaxis, ...], 100, axis=0) for img in img_tubulin]

print(f"Shape of first time stack: {img_actin_series[0].shape}")  # Should print (100, height, width)
print(f"Shape of first time stack: {img_tubulin_series[0].shape}")  # Should print (100, height, width)


In [ ]:
# Plot raw and denoised image next to each other

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(img_actin_series[0][0])
ax[1].imshow(img_tubulin_series[0][0])

## Creating composites of denoised, raw, actin and tubulin images

In [ ]:
# Create composite images of denoised and raw images

img_composite = []

for i in tqdm(range(len(img_denoised))):
    # Load the corresponding frames from each image series
    image1 = img_denoised[i]
    image2 = img_raw[i]
    image3 = img_actin_series[i]
    image4 = img_tubulin_series[i]
    
    # Create an empty composite image
    composite_image = np.zeros((100, 4, 1024, 1024), dtype=np.uint16)

    # Assign each image to a separate channel
    composite_image[:, 0, :, :] = image1
    composite_image[:, 1, :, :] = image2
    composite_image[:, 2, :, :] = image3
    composite_image[:, 3, :, :] = image4

    img_composite.append(composite_image)

img_composite[0].shape

In [34]:
# Display merged image

#fig, ax = plt.subplots(1, 4, figsize=(12, 4))

#for i in range(4):
    #ax[i].imshow(img_composite[i][0], cmap='gray')  # Display the first slice of each channel
    #ax[i].set_title(f"Channel {i+1}")
    #ax[i].axis('off')  # Turn off axis for cleaner visualization

#plt.tight_layout()
#plt.show()

In [35]:
# Initializes Napari viewer

#viewer = napari.Viewer()

In [36]:
# Adds image to Napari viewer

#viewer.add_image(img_composite[0])

In [ ]:
print(img_raw_files[0])

In [ ]:
# Save merged images

for img, tiff_file in zip(img_composite, img_raw_files):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    print(file_name)
    
    # Define the output file path
    output_file = os.path.join(img_composite_dir, file_name + '_merged.tif')
    print(output_file)
    
    # Save the manipulated image as a TIFF
    tf.imwrite(output_file, img, metadata={"axes": "TCYX"}, imagej=True)